# 🧠 BRISC 2025 Brain Tumor Segmentation — Pretrained Vision Transformer U-Net (2D)

Ce notebook implémente un **Vision Transformer hiérarchique avec décodeur U-Net** (SegFormer / `mit_b2` pré-entraîné sur **ImageNet-1k**) en 2D pour la détection et la segmentation des tumeurs cérébrales (Gliome, Méningiome, Pituitaire) sur le dataset **BRISC 2025**.

### 🌟 Points clés de l'architecture :
- **Encodeur Vision Transformer Pré-entraîné (ImageNet)** : Initialisé avec les poids officiels `mit_b2` via `segmentation_models_pytorch` pour une extraction de caractéristiques riche et une convergence accélérée.
- **Décodeur CNN U-Net hiérarchique** : Skip connections directes pour préserver les résolutions fines et délimiter précisément les contours tumoraux.
- **Dataset nettoyé & anti-fuite** : Intégration directe du split propre (`clean_manifest.csv`) issu de l'audit.
- **Perte combinée DiceCELoss** : Équilibre optimal pour classes déséquilibrées (tumeurs vs fond).


In [1]:
pip install monai loguru kagglehub scikit-learn segmentation-models-pytorch timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import kagglehub

DATA_DIR = kagglehub.dataset_download("briscdataset/brisc2025")
print(f"Dataset downloaded to: {DATA_DIR}")

Dataset downloaded to: /kaggle/input/datasets/briscdataset/brisc2025


In [3]:
import glob
import time
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from loguru import logger
from monai.networks.nets import SwinUNETR
import segmentation_models_pytorch as smp
from monai.networks.layers import Norm
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.utils import one_hot
from monai.transforms import (
    Compose,
    RandAffined,
    RandAdjustContrastd,
    RandFlipd,
    EnsureTyped,
    CastToTyped,
    NormalizeIntensityd,
)

# Configuration
CONFIG = {
    "data_dir": DATA_DIR,
    "batch_size": 16,
    "image_size": (512, 512),
    "num_epochs_seg": 50,
    "learning_rate_seg": 1e-4,
    "encoder_name": "mit_b2",  # Hierarchical Vision Transformer (SegFormer)
    "encoder_weights": "imagenet",  # Official ImageNet-1k pre-trained weights
    "seg_in_channels": 3,
    "seg_out_channels": 2,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


print(f"Device: {CONFIG['device']}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

2026-10-15 12:32:44.655374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1792067564.801059      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1792067564.843464      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1792067565.209696      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792067565.209737      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792067565.209740      58 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla T4
VRAM: 14.6 GB


In [4]:
from scipy import ndimage as ndi


class BRISCSegDataset(Dataset):
    """
    PyTorch Dataset for the BRISC 2025 segmentation task.
    Supports direct path injection, so the clean train/val pairs produced by the audit manifest
    can be plugged in, and optional load-time mask repair: masks whose stem is listed in
    `fill_holes_stems` get their interior holes filled.
    """
    def __init__(self, image_dir=None, mask_dir=None, image_size=(512, 512), augment=False,
                 fill_holes_stems=None):
        self.image_size = image_size
        self.augment = augment
        self.fill_holes_stems = set(fill_holes_stems or [])
        self.pairs = []

        # Only search for files if directories are provided
        if image_dir is not None and mask_dir is not None:
            image_path_obj = Path(image_dir)
            mask_path_obj = Path(mask_dir)
            for img_path in sorted(image_path_obj.glob("*.jpg")):
                mask_path = mask_path_obj / (img_path.stem + ".png")
                if mask_path.exists():
                    self.pairs.append((str(img_path), str(mask_path)))

        self.aug_transform = Compose([
            NormalizeIntensityd(keys=["image"], channel_wise=True),
            RandAdjustContrastd(keys=["image"], prob=0.3, gamma=(0.7, 1.3)),
            RandAffined(
                keys=["image", "label"],
                prob=0.5,
                rotate_range=(0.26, 0.26),
                scale_range=(0.1, 0.1),
                mode=("bilinear", "nearest"),
                padding_mode="zeros"
            ),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            CastToTyped(keys=["label"], dtype=np.int64),
            EnsureTyped(keys=["image", "label"])
        ])

        self.val_transform = Compose([
            NormalizeIntensityd(keys=["image"], channel_wise=True),
            CastToTyped(keys=["label"], dtype=np.int64),
            EnsureTyped(keys=["image", "label"])
        ])

    def __len__(self):
        return len(self.pairs)

    def _load_image(self, path):
        img = Image.open(path).convert("RGB")
        img = img.resize(self.image_size, Image.BILINEAR)
        return np.array(img, dtype=np.float32).transpose(2, 0, 1)

    def _load_mask(self, path):
        mask = Image.open(path).convert("L")
        mask = mask.resize(self.image_size, Image.NEAREST)
        binary = np.array(mask) > 127
        if Path(path).stem in self.fill_holes_stems:      # audit flag MASK_HOLES -> repair
            binary = ndi.binary_fill_holes(binary)
        return binary.astype(np.float32)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        data_dict = {
            "image": self._load_image(img_path),
            "label": self._load_mask(mask_path)[None, ...]
        }

        if self.augment:
            data_dict = self.aug_transform(data_dict)
        else:
            data_dict = self.val_transform(data_dict)

        return data_dict["image"], data_dict["label"]

In [5]:
import pandas as pd

# Path to clean dataset manifest (change this path whenever needed)
MANIFEST_PATH = "/kaggle/input/datasets/rabiaeibenltaief/cleaning/clean_manifest.csv"

def discover_segmentation_data(data_dir):
    data_path = Path(data_dir)
    seg_dirs = list(data_path.rglob("segmentation_task"))
    if not seg_dirs:
        raise FileNotFoundError(f"Could not find 'segmentation_task' in {data_dir}")
    seg_root = seg_dirs[0]
    train_img = seg_root / "train" / "images"
    train_mask = seg_root / "train" / "masks"
    test_img = seg_root / "test" / "images"
    test_mask = seg_root / "test" / "masks"
    return str(train_img), str(train_mask), str(test_img), str(test_mask)


def build_clean_splits(train_img_dir, train_mask_dir, manifest_path=MANIFEST_PATH):
    """
    Builds the clean, leakage-aware train/val pairs directly from clean_manifest.csv:
      1. drops every sample the audit excluded (exact duplicates, duplicates with conflicting masks,
         exact copies of test images),
      2. uses the audit's group-aware train/val folds: stratified by tumor type x plane, with near-duplicate groups
         kept on one side so near-identical slices never straddle train and val.
    """
    manifest = pd.read_csv(manifest_path)
    img_dir, mask_dir = Path(train_img_dir), Path(train_mask_dir)

    if "task" in manifest.columns:
        rows = manifest[(manifest["split"] == "train") & (manifest["task"] == "segmentation")].copy()
    else:
        rows = manifest[manifest["split"] == "train"].copy()

    rows["image_path"] = [str(img_dir / f"{s}.jpg") for s in rows["stem"]]
    rows["mask_path"] = [str(mask_dir / f"{s}.png") for s in rows["stem"]]


    n_raw = len(rows)

    # 1. Exclusions decided by the audit
    excluded = rows[rows["fold"] == "excluded"]
    reasons = excluded["exclude_reason"].dropna().str.split(";").explode().value_counts()
    rows = rows[rows["fold"].isin(["train", "val"])]

    # 2. Safety check: no near-duplicate group spans train and val
    sides = rows.groupby("group")["fold"].nunique()
    if (sides > 1).any():
        raise ValueError(f"{int((sides > 1).sum())} near-duplicate groups span train and val.")

    train_rows = rows[rows["fold"] == "train"]
    val_rows = rows[rows["fold"] == "val"]
    train_pairs = list(zip(train_rows["image_path"], train_rows["mask_path"]))
    val_pairs = list(zip(val_rows["image_path"], val_rows["mask_path"]))

    print(f"\n{'=' * 60}")
    print(" DATASET CLEANING REPORT ")
    print(f"{'=' * 60}")
    print(f"  • Raw train images audited:               {n_raw}")
    print(f"  • Excluded by the audit:                  {len(excluded)}")
    for reason, n in reasons.items():
        print(f"      - {reason:<34} {n}")
    print(f"  • Clean pairs retained:                   {len(rows)}  ->  {len(train_pairs)} train / {len(val_pairs)} val")
    print("  • No near-duplicate group spans train and val ✓")
    print(f"{'=' * 60}")
    print(pd.crosstab(rows["tumor"], rows["fold"]).to_string())
    print()
    return train_pairs, val_pairs, rows


train_img_dir, train_mask_dir, test_img_dir, test_mask_dir = discover_segmentation_data(CONFIG["data_dir"])

train_pairs, val_pairs, clean_df = build_clean_splits(train_img_dir, train_mask_dir, manifest_path=MANIFEST_PATH)

train_seg = BRISCSegDataset(image_size=CONFIG["image_size"], augment=True)
train_seg.pairs = train_pairs

val_seg = BRISCSegDataset(image_size=CONFIG["image_size"], augment=False)
val_seg.pairs = val_pairs

print(f"Clean split: {len(train_pairs)} train / {len(val_pairs)} val")

# DataLoaders
train_loader = DataLoader(
    train_seg,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_seg,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)



 DATASET CLEANING REPORT 
  • Raw train images audited:               3933
  • Excluded by the audit:                  48
      - conflicting_mask                   35
      - exact_duplicate                    13
      - test_leak                          9
  • Clean pairs retained:                   3885  ->  3108 train / 777 val
  • No near-duplicate group spans train and val ✓
fold        train  val
tumor                 
glioma        917  230
meningioma   1044  261
pituitary    1147  286

Clean split: 3108 train / 777 val


In [6]:
def train_segmentation():
    """Train 2D Swin UNETR on BRISC 2025 segmentation task."""
    device = CONFIG["device"]
    print(f"\n{'='*60}")
    print(" TRAINING 2D PRETRAINED VISION TRANSFORMER (SegFormer/U-Net)")
    print(f"{'='*60}")

    # Model (2D Vision Transformer U-Net with ImageNet Pretrained Weights)
    model = smp.Unet(
        encoder_name=CONFIG.get("encoder_name", "mit_b2"),
        encoder_weights=CONFIG.get("encoder_weights", "imagenet"),
        in_channels=CONFIG["seg_in_channels"],
        classes=CONFIG["seg_out_channels"],
    ).to(device)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Loss, optimizer
    class_weights = torch.tensor([1.0, 5.0]).to(device)

    loss_fn = DiceCELoss(
        to_onehot_y=True,
        softmax=True,
        squared_pred=True,
        lambda_dice=0.7,
        lambda_ce=0.3,
        weight=class_weights,
    ).to(device)

    # Optimizer & Learning Rate Scheduler (Cosine Annealing schedule aligned with classification)
    from torch.optim.lr_scheduler import CosineAnnealingLR

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG["learning_rate_seg"], weight_decay=1e-4
    )
    scheduler = CosineAnnealingLR(
        optimizer, T_max=CONFIG["num_epochs_seg"], eta_min=1e-6
    )

    dice_metric = DiceMetric(include_background=False, reduction="mean")
    scaler = torch.amp.GradScaler("cuda")

    best_dice = 0.0
    no_improve = 0
    train_losses, val_dices = [], []

    for epoch in range(CONFIG["num_epochs_seg"]):
        model.train()
        epoch_loss, steps = 0, 0

        for images, masks in train_loader:
            steps += 1
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(images)
                loss = loss_fn(preds, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["grad_clip_norm"])
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        epoch_loss /= max(steps, 1)
        train_losses.append(epoch_loss)

        # Learning rate schedule step
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # --- Validate every 5 epochs ---
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                for val_imgs, val_masks in val_loader:
                    val_imgs = val_imgs.to(device)
                    val_masks = val_masks.to(device).long()

                    with torch.amp.autocast("cuda"):
                        val_preds = model(val_imgs)
                        val_preds = torch.softmax(val_preds, dim=1)

                    pred_oh = one_hot(torch.argmax(val_preds, dim=1, keepdim=True), CONFIG["seg_out_channels"])
                    mask_oh = one_hot(val_masks, CONFIG["seg_out_channels"])
                    dice_metric(pred_oh, mask_oh)

            mean_dice = dice_metric.aggregate().item()
            dice_metric.reset()
            val_dices.append(mean_dice)

            print(f"Epoch {epoch+1:3d}/{CONFIG['num_epochs_seg']} | Loss: {epoch_loss:.4f} | Dice: {mean_dice:.4f} | LR: {current_lr:.2e}")

            if mean_dice > best_dice:
                best_dice = mean_dice
                no_improve = 0
                torch.save(model.state_dict(), "segmentation_swinunetr.pth")
                print(f"→ Best Pretrained ViT model saved! Dice: {best_dice:.4f}")
            else:
                no_improve += 5

            if no_improve >= CONFIG["early_stop_patience"]:
                print(f"Early stopping (no improvement for {no_improve} epochs)")
                break
        else:
            print(f"Epoch {epoch+1:3d}/{CONFIG['num_epochs_seg']} | Loss: {epoch_loss:.4f} | LR: {current_lr:.2e}")

    print(f"\nTraining complete. Best Dice: {best_dice:.4f}")
    return model, train_losses, val_dices


In [7]:
# Train segmentation
seg_model, train_losses, val_dices = train_segmentation()


 TRAINING 2D PRETRAINED VISION TRANSFORMER (SegFormer/U-Net)


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

Model parameters: 27,477,154
Epoch   1/50 | Loss: 0.8044 | LR: 9.99e-05
Epoch   2/50 | Loss: 0.3092 | LR: 9.96e-05
Epoch   3/50 | Loss: 0.2445 | LR: 9.91e-05
Epoch   4/50 | Loss: 0.2178 | LR: 9.84e-05
Epoch   5/50 | Loss: 0.1994 | Dice: 0.8637 | LR: 9.76e-05
→ Best Pretrained ViT model saved! Dice: 0.8637
Epoch   6/50 | Loss: 0.1821 | LR: 9.65e-05
Epoch   7/50 | Loss: 0.1696 | LR: 9.53e-05
Epoch   8/50 | Loss: 0.1598 | LR: 9.39e-05
Epoch   9/50 | Loss: 0.1533 | LR: 9.23e-05
Epoch  10/50 | Loss: 0.1452 | Dice: 0.8747 | LR: 9.05e-05
→ Best Pretrained ViT model saved! Dice: 0.8747
Epoch  11/50 | Loss: 0.1410 | LR: 8.86e-05
Epoch  12/50 | Loss: 0.1326 | LR: 8.66e-05
Epoch  13/50 | Loss: 0.1237 | LR: 8.44e-05
Epoch  14/50 | Loss: 0.1204 | LR: 8.21e-05
Epoch  15/50 | Loss: 0.1153 | Dice: 0.8778 | LR: 7.96e-05
→ Best Pretrained ViT model saved! Dice: 0.8778
Epoch  16/50 | Loss: 0.1086 | LR: 7.70e-05
Epoch  17/50 | Loss: 0.1072 | LR: 7.43e-05
Epoch  18/50 | Loss: 0.1026 | LR: 7.16e-05
Epoch  1